# Ube Craze Sentiment Visualizations

This notebook generates visualizations from the final sentiment-scored dataset to illustrate the dynamics of Filipino gastronationalism (positive pride) versus potential cultural dilution (negative/neutral/exoticizing themes).

## 1. Imports and Config

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from ube_craze_nlp.nlp.clean import tokenize_text, remove_stopwords
from ube_craze_nlp.utils.paths import FINAL_DATA_DIR, OUTPUTS_DIR, ensure_dirs

ensure_dirs()

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

print(f"Figures will be exported to: {OUTPUTS_DIR}")

## 2. Load Scored Dataset

In [ ]:
final_file = FINAL_DATA_DIR / "sentiment_scored_comments.csv"

if not final_file.exists():
    print("❌ Scored dataset not found! Please run Notebook 03 first.")
    df_comments = pd.DataFrame()
else:
    df_comments = pd.read_csv(final_file)
    print(f"Loaded {len(df_comments)} scored comments.")
    display(df_comments.head())

## 3. Visualize Sentiment Distribution

We plot the overall sentiment distribution (Positive, Neutral, Negative) to understand public reception of the Ube craze on TikTok.

In [ ]:
if not df_comments.empty:
    # Compute counts and percentages
    sentiment_counts = df_comments["sentiment"].value_counts()
    sentiment_pct = df_comments["sentiment"].value_counts(normalize=True) * 100

    # Plot
    palette = {"positive": "#6a0dad", "neutral": "#a295a7", "negative": "#d46a83"}  # Purple, Grey, Muted Pink
    ax = sns.barplot(
        x=sentiment_counts.index,
        y=sentiment_counts.values,
        hue=sentiment_counts.index,
        palette=palette,
        legend=False
    )

    plt.title("TikTok Comment Sentiment on the Global Ube Craze", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Sentiment Label", fontsize=12)
    plt.ylabel("Number of Comments", fontsize=12)

    # Add text labels on top of the bars
    for i, p in enumerate(ax.patches):
        height = p.get_height()
        percentage = sentiment_pct.iloc[i]
        ax.annotate(
            f"{int(height)} ({percentage:.1f}%)",
            (p.get_x() + p.get_width() / 2., height),
            ha="center", va="center",
            xytext=(0, 9), textcoords="offset points",
            fontsize=11, fontweight="semibold"
        )

    sns.despine()
    plt.tight_layout()

    # Save plot
    fig_path = OUTPUTS_DIR / "sentiment_distribution.png"
    plt.savefig(fig_path, dpi=300)
    print(f"✅ Exported plot to: {fig_path}")
    plt.show()

## 4. Word Frequency Distributions (Pride vs. Exoticization)

We extract high-frequency terms from positive comments (potential indicators of national pride and Gastronationalism) and compare them to terms from neutral or negative comments (potential indicators of commercialization, exoticization, or cultural dilution).

In [ ]:
def get_frequent_words(texts_series, top_n=15):
    all_tokens = []
    for text in texts_series:
        tokens = tokenize_text(str(text))
        cleaned_tokens = remove_stopwords(tokens)
        all_tokens.extend(cleaned_tokens)
    return Counter(all_tokens).most_common(top_n)

if not df_comments.empty:
    # Separate positive and neutral/negative comment texts
    pos_comments = df_comments[df_comments["sentiment"] == "positive"]["normalized_text"]
    neg_neu_comments = df_comments[df_comments["sentiment"].isin(["negative", "neutral"])] ["normalized_text"]

    # Compute word counts
    pos_words = get_frequent_words(pos_comments)
    neg_neu_words = get_frequent_words(neg_neu_comments)

    # --- Plot Positive Words ---
    df_pos_words = pd.DataFrame(pos_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_pos_words, x="count", y="word", hue="word", palette="Purples_r", legend=False)
    plt.title("Top Words in Positive Comments (Cultural Pride / Gastronationalism)", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    pos_fig_path = OUTPUTS_DIR / "word_frequency_pride.png"
    plt.savefig(pos_fig_path, dpi=300)
    print(f"✅ Exported positive words plot to: {pos_fig_path}")
    plt.show()

    # --- Plot Neutral/Negative Words ---
    df_neg_words = pd.DataFrame(neg_neu_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_neg_words, x="count", y="word", hue="word", palette="Reds_r", legend=False)
    plt.title("Top Words in Neutral & Negative Comments (Commercialization / Exoticization / Context)", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    neg_fig_path = OUTPUTS_DIR / "word_frequency_exotic.png"
    plt.savefig(neg_fig_path, dpi=300)
    print(f"✅ Exported neutral/negative words plot to: {neg_fig_path}")
    plt.show()